# Étape 3 : Embeddings

- Objectif : convertir du texte en vecteurs numériques (embeddings) pour pouvoir :
    - faire de la recherche par similarité (cosine similarity)
    - alimenter une base vectorielle (ex: Chroma)

#### Requirements (Embeddings - Local)

Dépendances Python :
- langchain
- langchain-community
- chromadb (pour l’étape suivante)
- requests (souvent utile)

Pré-requis système :
- Ollama installé et lancé
- Un modèle d’embeddings disponible (recommandé : `mxbai-embed-large`)

👉 Commandes utiles (terminal) :
- ollama --version
- ollama pull mxbai-embed-large


In [ ]:
%pip install -U langchain langchain-community chromadb requests


## Vérification : Ollama tourne ?

On teste si l’API locale répond (par défaut sur http://localhost:11434).
Si cela échoue : lance `ollama serve` ou ouvre l’app Ollama.

In [8]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    r.raise_for_status()
    print("✅ Ollama est accessible.")
    print("Modèles disponibles :", [m["name"] for m in r.json().get("models", [])][:10])
except Exception as e:
    print("❌ Ollama n'est pas accessible. Détail :", e)


✅ Ollama est accessible.
Modèles disponibles : ['gemma3:12b', 'nomic-embed-text:latest', 'deepseek-r1:1.5b', 'deepseek-r1:latest', 'llama3:latest', 'llama2:latest', 'mxbai-embed-large:latest', 'gemma2:2b']


### 1. OllamaEmbeddings (Local)
---

`Recommandation` :
Utiliser un vrai modèle d'embeddings, par ex :
- `mxbai-embed-large` (très utilisé)
- `nomic-embed-text`

⚠️ Note :
Un modèle LLM (ex: `gemma2:2b`) n’est pas forcément un modèle embeddings.
Pour un TP RAG propre → utiliser un **embedding model**.


In [12]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="mxbai-embed-large")

####  Tester `embed_documents`

`embed_documents()` prend une liste de textes et retourne une liste de vecteurs.
Chaque vecteur = liste de floats.


In [13]:
r1 = embeddings.embed_documents([
    "Alpha est la première lettre de l'alphabet grec.",
    "Beta est la deuxième lettre de l'alphabet grec."
])

print("Nombre de documents vectorisés :", len(r1))
print("Dimension du vecteur (doc 0) :", len(r1[0]))
print("Extrait vecteur (doc 0) :", r1[0][:10])

Nombre de documents vectorisés : 2
Dimension du vecteur (doc 0) : 1024
Extrait vecteur (doc 0) : [0.023419130593538284, 0.00450938381254673, 0.03667385131120682, 0.02587188221514225, -0.05989900603890419, -0.014161794446408749, 0.04001428559422493, -0.01500067301094532, -0.01117914542555809, 0.052519623190164566]


#### Tester `embed_query`

`embed_query()` retourne un seul vecteur correspondant à une requête.


In [14]:
q_vec = embeddings.embed_query("Quelle est la deuxième lettre de l'alphabet grec ?")

print("Dimension du vecteur requête :", len(q_vec))
print("Extrait vecteur requête :", q_vec[:10])

Dimension du vecteur requête : 1024
Extrait vecteur requête : [-0.006265810690820217, -0.01277928426861763, 0.017019664868712425, 0.008512936532497406, -0.035153478384017944, 0.0018819888355210423, 0.053922612220048904, -0.02249358408153057, -0.002129241358488798, 0.04057788848876953]


### 2. Pipeline complet : Loader → Splitter → Embeddings-ready
---

On reprend :
- `speech.txt` (TextLoader)
- découpage en chunks (RecursiveCharacterTextSplitter)
- résultat : `final_documents` prêt à être indexé dans Chroma


In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("speech.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_documents = splitter.split_documents(docs)

print("Docs initiaux :", len(docs))
print("Chunks finaux :", len(final_documents))
print("\nExtrait chunk 0 :\n", final_documents[0].page_content[:300])

Docs initiaux : 1
Chunks finaux : 2

Extrait chunk 0 :
 Dans la douce brise de Joal, un jeune garçon nommé Léopold écoutait les chants anciens de son peuple Sérère. 
Ces mélodies murmuraient des histoires de rois et de baobabs sacrés.
Des années plus tard, loin des rivages du Sénégal et sous le ciel gris de Paris, 
il transforma ces souvenirs en poésie. 


### 3. Autres embedding models Ollama
---
## 5) Autres modèles embeddings (Ollama)

Exemples :
- nomic-embed-text
- mxbai-embed-large

Voir : https://ollama.com/blog/embedding-models

In [16]:
embeddings_alt = OllamaEmbeddings(model="nomic-embed-text")
vec = embeddings_alt.embed_query("Ceci est un document de test.")
print("Dimension :", len(vec))

Dimension : 768


## Take Away (Embeddings)

- Les embeddings transforment le texte → vecteurs numériques.
- Ces vecteurs servent à la **similarity search** (cosine similarity).
- En local : `OllamaEmbeddings(model="mxbai-embed-large")` est un excellent choix.
- Prochaine étape : **Vector Store Chroma** + indexation + similarity_search.